# Step 0 · Validation of the SBERT filter on PROD

Measures how good the final dataset is, on the selected run (the historical comparison below used **PROD**): **precision** (share of the kept families that are drones) and **recall** (share of the drones found by the search that the filter keeps).

- **Input:** `df_sbert_scored.parquet`, written by the data preparation notebook (part A) and scored on Colab (`1c-SbertScoring_Colab.ipynb`). The kept/rejected split is rebuilt here from `TAU_FILTER`.
- **Settings read from this run’s selection manifest:** `DRONE_DESCRIPTION` (the sector definition given to the LLM) and `TAU_FILTER`, so the two notebooks cannot drift apart.
- **Method** (`2-DataValidation`): random families from the kept and from the rejected group (seed 42), one abstract per family, labelled by gpt-4.1-mini against `DRONE_DESCRIPTION`; `incerto` answers are labelled by hand (section 3); precision and recall with 95% Wilson intervals.
- **Cost:** about 5 cents per 100 LLM calls; families already labelled with the same definition are read from the cache.
- **Output:** `validation/prod_filter_validation_v2.csv` (numbers only) and the hand labels in `validation/review_<run>_*.csv`. Section 5 exports compact review tables in the run’s `validation/export/` directory.

History: the first PROD validation (10 Nov; earlier definition, threshold 0.45, abstracts only) gave precision 0.88 (95% CI 0.80–0.93) and recall 0.94; its files are `prod_filter_validation.csv` and `review_filtro_sbert_prod_*.csv`, left unchanged.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "sector.yaml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from run_baseline import Run

r = Run.open(ROOT)
from validation_state import fill_labels, input_fingerprints
import json
import re

import pandas as pd
from IPython.display import display

if r.env != "PROD":
    raise ValueError("PROD filter validation requires a PROD run.")

SCORED_REL = "preparation/df_sbert_scored.parquet"
SCORED_FILE = r.verify(SCORED_REL)
N_KEPT = 100                          # random families checked among the kept ones
N_REJECTED = 100                      # and among the rejected ones
RUN_NAME = "Filtro SBERT PROD v2"     # review files: validation/review_filtro_sbert_prod_v2_*.csv
RESULT_FILE = "prod_filter_validation_v2.csv"   # the first PROD run (10 Nov) stays in prod_filter_validation.csv

# Use the settings recorded when 1b produced this run's selection.
selection_manifest = json.loads(r.verify("manifest.json").read_text(encoding="utf-8"))
if (
    selection_manifest["inputs"][SCORED_REL] != r.receipt(SCORED_REL)["sha256"]
    or selection_manifest["run_id"] != r.run_id or selection_manifest["env"] != r.env
):
    raise ValueError("The selected manifest does not identify this run and scored input.")
DRONE_DESCRIPTION = selection_manifest["sector_description"]
TAU_FILTER = float(selection_manifest["threshold"])

print(f"Selection manifest: {r.directory / 'manifest.json'} | TAU_FILTER = {TAU_FILTER}")
print("Definition starts with:", DRONE_DESCRIPTION.splitlines()[0])

## 1. Kept and rejected families

A family is **kept** if at least one of its applications reaches the threshold; it is **rejected** if none does.

In [ ]:
scored = pd.read_parquet(SCORED_FILE)
scored = scored.dropna(subset=["docdb_family_id"]).copy()
scored["docdb_family_id"] = scored["docdb_family_id"].astype("int64")
scored = scored[scored["docdb_family_id"] > 0]

kept = scored[scored["sbert_similarity"] >= TAU_FILTER].copy()
kept_ids = set(kept["docdb_family_id"])
rejected = scored[(scored["sbert_similarity"] < TAU_FILTER)
                  & ~scored["docdb_family_id"].isin(kept_ids)].copy()

assert kept_ids.isdisjoint(set(rejected["docdb_family_id"]))
print(f"Applications scored: {len(scored):,} | kept: {len(kept):,} ({len(kept) / len(scored):.1%})")
print(f"Families kept: {kept['docdb_family_id'].nunique():,} | rejected: {rejected['docdb_family_id'].nunique():,}")

## 2. LLM labels

In [ ]:
%pip install -q langchain-core langchain-openai pydantic
VALIDATOR = ROOT / "Step0" / "2-DataValidation.ipynb"
%run -i "$VALIDATOR"

pid = prompt_id(DRONE_DESCRIPTION)
already = sum(1 for (family, p) in load_llm_cache() if p == pid)
print(f"Prompt id {pid}: {already} families already labelled with this definition (reused from the cache)")

validation_kept = validate_sample(kept, n=N_KEPT)
validation_rejected = validate_sample(rejected, n=N_REJECTED)

slug = re.sub(r"[^a-z0-9]+", "_", RUN_NAME.lower()).strip("_")
REVIEW_FILES = {tag: VALIDATION_DIR / f"review_{slug}_{tag}.csv" for tag in ("selezionati", "scartati")}

label_state = input_fingerprints(LLM_CACHE, *REVIEW_FILES.values())
prod_metrics = evaluate_selection(
    validation_selected=validation_kept,
    validation_rejected=validation_rejected,
    df_selected=kept,
    df_rejected=rejected,
    name=RUN_NAME,
)
if not prod_metrics.get("to_check_by_hand", 0):
    # Keep only family identifiers and labels as evidence for the result.
    evaluated_labels = []
    for tag, sample in [("selezionati", validation_kept), ("scartati", validation_rejected)]:
        resolved = _resolved_labels(sample, REVIEW_FILES[tag])
        for family, raw, final in zip(sample["docdb_family_id"], sample["llm_label"], resolved):
            evaluated_labels.append({"group": tag, "family": int(family), "llm_label": raw, "resolved_label": final})
    if input_fingerprints(LLM_CACHE, *REVIEW_FILES.values()) != label_state:
        raise ValueError("Validation inputs changed during evaluation. Rerun the label and evaluation cell.")
display(prod_metrics.to_frame().round(3))

## 3. Hand labels for the `incerto` cases

The first cell shows the families the LLM could not decide. Read them, fill `LABELS` in the second cell (`"pertinente"` or `"non_pertinente"`, with the same definition as the LLM: `DRONE_DESCRIPTION`), run it, then **Restart Kernel and Run All**: the LLM is not called again.

In [ ]:
for tag, path in REVIEW_FILES.items():
    if not path.exists():
        continue
    review = pd.read_csv(path, keep_default_na=False)
    group = "KEPT" if tag == "selezionati" else "REJECTED"
    for _, row in review.iterrows():
        print(f"\n[{row['docdb_family_id']}] ({group}) {row.get('appln_title', '')}\n  {row.get('appln_abstract', '')}\n"
              f"  LLM: {row['reason']}\n  hand label: {row['human_label'] or '-'}")

In [ ]:
LABELS = {
    # docdb_family_id: "pertinente" or "non_pertinente"
}

assert set(LABELS.values()) <= {"pertinente", "non_pertinente"}, "Use only pertinente / non_pertinente"
for path in REVIEW_FILES.values():
    if path.exists():
        review = fill_labels(path, LABELS)
        print(path.name, "| still empty:", int((review["human_label"].astype(str).str.strip() == "").sum()))

## 4. Results

- **precision:** share of the kept families that are drones; **estimated drone families kept** = kept families × precision.
- **relevant_share_in_rejected:** share of drones among the rejected families; **estimated drone families lost** = rejected families × that share.
- **recall_within_pool:** drones kept / drones found by the search (the search itself can miss drones: this is recall *within* the search results).

Then the errors: non-drones among the kept families (they lower precision) and drones among the rejected ones (they lower recall). Shown here only (titles are not saved), to describe the errors in the appendix.

In [ ]:
if prod_metrics.get("to_check_by_hand", 0):
    print("Label the 'incerto' cases first (section 3), then re-run.")
else:
    if input_fingerprints(LLM_CACHE, *REVIEW_FILES.values()) != label_state:
        raise ValueError("Validation inputs changed after evaluation. Rerun the label and evaluation cell.")
    summary = prod_metrics.astype(float)
    summary["estimated_drone_families_kept"] = summary["selected_families"] * summary["precision"]
    summary["estimated_drone_families_lost"] = (summary["rejected_families"]
                                                * summary["relevant_share_in_rejected"])
    summary = summary.round(4)
    summary["tau_filter"] = TAU_FILTER
    summary["prompt_id"] = pid
    r.publish(
        f"validation/{RESULT_FILE}", lambda path: summary.to_frame().T.to_csv(path, index=False),
        inputs=("manifest.json", SCORED_REL),
        metadata={"label_inputs_sha256": label_state, "sample_labels": evaluated_labels},
    )
    display(summary.to_frame())
    print("Saved:", VALIDATION_DIR / RESULT_FILE)

    for title, sample, tag, wrong in [
        ("Kept but not drones", validation_kept, "selezionati", "non_pertinente"),
        ("Rejected but drones", validation_rejected, "scartati", "pertinente"),
    ]:
        labels = _resolved_labels(sample, REVIEW_FILES[tag])
        errors = sample.reset_index(drop=True)[labels.eq(wrong)]
        print(f"\n{title}: {len(errors)} of {len(sample)}")
        display(errors[["docdb_family_id", "sbert_similarity", "appln_title", "reason"]])

## 5. Export adjudicated review tables

Publish the compact review for this run after adjudication. The working CSVs retain their titles, abstracts and human edits. Edit those files directly to revise a filled label.

In [ ]:
KEEP = ["docdb_family_id", "llm_label", "reason", "human_label"]
for path in REVIEW_FILES.values():
    if not path.exists():
        continue
    review = pd.read_csv(path, keep_default_na=False)
    if review["human_label"].astype(str).str.strip().eq("").any():
        print(f"{path.name}: fill every hand label before export")
        continue
    r.publish(
        f"validation/export/{path.name}", lambda dest: review[KEEP].to_csv(dest, index=False),
        inputs=("manifest.json", SCORED_REL),
        metadata={"review_sha256": input_fingerprints(path)},
    )
    print(f"Exported {len(review)} labelled rows from {path.name}")